# High-level developer example: Seurat CCA + RPCA on Dataset 0

This notebook runs **Seurat v5 CCAIntegration and RPCAIntegration** on scRareBench **Dataset 0 (GSE194122 paper benchmark)** and evaluates both latent spaces with the high-level `MethodSpec → benchmark_method` API.

Seurat remains user-owned method code. scRareBench provides the dataset, validates cell alignment, runs the benchmark, and writes the result artifacts.

The Seurat preprocessing used here follows the existing benchmark notebook: raw GEX counts → `NormalizeData` → `FindVariableFeatures(nfeatures=3000)` → `ScaleData` → `RunPCA(npcs=50)` → integration with dimensions 1:30.

The default is one method seed (`42`) to keep a full Dataset 0 Colab run practical. You can add more seeds if needed.

In [ ]:
# Optional diagnostic only: leave commented unless you want to inspect the runtime.
# import sys, subprocess
# from importlib import metadata
# print(sys.version)
# print(subprocess.check_output(["Rscript", "--version"], stderr=subprocess.STDOUT, text=True))
# for pkg in ("numpy", "pandas", "scipy", "anndata", "scanpy", "scrarebench"):
#     try: print(pkg, metadata.version(pkg))
#     except metadata.PackageNotFoundError: print(pkg, "not installed")


In [ ]:
import subprocess, sys

REPO = "git+https://github.com/amirhossein-alishahi/scRareBench_.git@v0.10.5"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", REPO])

from scrarebench.runtime import setup_runtime
runtime_report = setup_runtime(quiet=False)


In [ ]:
# Install R (if needed) and Seurat v5.
# Seurat is a method dependency, so it is intentionally installed outside scRareBench.
import os, shutil, subprocess

env = os.environ.copy()
env["DEBIAN_FRONTEND"] = "noninteractive"
subprocess.check_call(["apt-get", "update", "-qq"], env=env)
subprocess.check_call([
    "apt-get", "install", "-y", "-qq",
    "r-base", "build-essential", "gfortran",
    "libcurl4-openssl-dev", "libssl-dev", "libxml2-dev",
    "libhdf5-dev", "libfontconfig1-dev", "libfreetype6-dev",
    "libpng-dev", "libtiff-dev", "libjpeg-dev",
    "libharfbuzz-dev", "libfribidi-dev", "libgit2-dev", "libglpk-dev"
], env=env)

if shutil.which("Rscript") is None:
    raise RuntimeError("Rscript was not installed successfully.")

r_install = r'''
options(repos = c(CRAN = "https://cloud.r-project.org"), timeout = 1200)
need_seurat <- !requireNamespace("Seurat", quietly = TRUE) ||
               packageVersion("Seurat") < package_version("5.0.0")
if (need_seurat) {
  install.packages("Seurat", dependencies = NA, Ncpus = 2)
}
if (!requireNamespace("Seurat", quietly = TRUE)) stop("Seurat installation failed.")
if (packageVersion("Seurat") < package_version("5.0.0")) stop("Seurat >= 5.0.0 is required.")
cat("Seurat version:", as.character(packageVersion("Seurat")), "\n")
cat("SeuratObject version:", as.character(packageVersion("SeuratObject")), "\n")
'''
subprocess.check_call(["Rscript", "-e", r_install])


In [ ]:
from scrarebench import load_dataset, dataset_info

adata = load_dataset(0)
info = dataset_info(adata)

assert info.get("dataset_index") == 0
assert info.get("batch_key") in adata.obs.columns
count_layer = info.get("count_layer") or "counts"
assert count_layer in adata.layers, f"Expected raw counts in adata.layers[{count_layer!r}]"

print(adata)
print("Dataset:", info.get("name"))
print("Batch key:", info.get("batch_key"))
print("Label key:", info.get("label_key"))
print("Count layer:", count_layer)


In [ ]:
# Reproducibility and Seurat controls.
METHOD_SEEDS = [42]   # e.g. [42, 123, 2026] for a multi-seed run
BENCHMARK_SEED = 42

METHOD_CONFIG = {
    "nfeatures": 3000,
    "npcs": 50,
    "integration_dims": 30,
}


## User-owned Seurat method section

The helper below is the only method-specific part. It:

1. takes the raw GEX count matrix from Dataset 0,
2. transfers counts + batch labels to R,
3. preprocesses once in Seurat,
4. runs RPCA and CCA on the same preprocessed object,
5. returns one cell × latent matrix per method with explicit barcode validation.

Ground-truth `celltype` labels are **not** passed to Seurat.

In [ ]:
from pathlib import Path
import shutil
import subprocess

import numpy as np
import pandas as pd
from scipy import io, sparse

from scrarebench import MethodOutput, dataset_info

SEURAT_CACHE = {}
SEURAT_ROOT = Path("/content/scrarebench_seurat_runtime")

R_SCRIPT = r'''
args <- commandArgs(trailingOnly = TRUE)
workdir <- args[[1]]
seed <- as.integer(args[[2]])
nfeatures <- as.integer(args[[3]])
npcs <- as.integer(args[[4]])
ndims <- as.integer(args[[5]])

suppressPackageStartupMessages({
  library(Seurat)
  library(Matrix)
})

if (packageVersion("Seurat") < package_version("5.0.0")) {
  stop("Seurat >= 5.0.0 is required.")
}
if (ndims > npcs) stop("integration_dims must be <= npcs")

set.seed(seed)
options(future.globals.maxSize = 20 * 1024^3)

counts <- Matrix::readMM(file.path(workdir, "counts.mtx"))
genes <- read.csv(file.path(workdir, "genes.csv"), stringsAsFactors = FALSE)
barcodes <- read.csv(file.path(workdir, "barcodes.csv"), stringsAsFactors = FALSE)
meta <- read.csv(file.path(workdir, "metadata.csv"), stringsAsFactors = FALSE, check.names = FALSE)

stopifnot(nrow(counts) == nrow(genes))
stopifnot(ncol(counts) == nrow(barcodes))
stopifnot(nrow(meta) == nrow(barcodes))
stopifnot("BATCH" %in% colnames(meta))

rownames(counts) <- make.unique(as.character(genes$gene))
colnames(counts) <- as.character(barcodes$barcode)
rownames(meta) <- as.character(meta$barcode)
meta <- meta[colnames(counts), , drop = FALSE]
stopifnot(all(rownames(meta) == colnames(counts)))
meta$BATCH <- factor(meta$BATCH)

obj <- CreateSeuratObject(
  counts = counts,
  meta.data = meta,
  assay = "RNA",
  project = "scRareBench_Dataset0_Seurat"
)
rm(counts)
gc()

obj[["RNA"]] <- split(obj[["RNA"]], f = obj$BATCH)
obj <- NormalizeData(obj, verbose = FALSE)
obj <- FindVariableFeatures(obj, nfeatures = nfeatures, verbose = FALSE)
obj <- ScaleData(obj, verbose = FALSE)
obj <- RunPCA(obj, npcs = npcs, verbose = FALSE)

save_latent <- function(object, reduction_name, output_name) {
  emb <- Embeddings(object[[reduction_name]])
  out <- data.frame(barcode = rownames(emb), emb, check.names = FALSE)
  write.csv(out, file.path(workdir, output_name), row.names = FALSE, quote = FALSE)
}

obj <- IntegrateLayers(
  object = obj,
  method = RPCAIntegration,
  orig.reduction = "pca",
  new.reduction = "integrated.rpca",
  dims = seq_len(ndims),
  verbose = FALSE
)
save_latent(obj, "integrated.rpca", "seurat_rpca_latent.csv")
gc()

obj <- IntegrateLayers(
  object = obj,
  method = CCAIntegration,
  orig.reduction = "pca",
  new.reduction = "integrated.cca",
  dims = seq_len(ndims),
  verbose = FALSE
)
save_latent(obj, "integrated.cca", "seurat_cca_latent.csv")
gc()

sink(file.path(workdir, "seurat_session_info.txt"))
print(sessionInfo())
sink()
'''


def _gex_mask(method_adata):
    for key in ("feature_types", "feature_type", "modality"):
        if key not in method_adata.var.columns:
            continue
        values = method_adata.var[key].astype(str).str.lower()
        mask = values.str.contains("gene expression|gex|rna", regex=True).to_numpy()
        if mask.any():
            return mask
    return np.ones(method_adata.n_vars, dtype=bool)


def _write_seurat_inputs(method_adata, workdir):
    meta = dataset_info(method_adata)
    batch_key = str(meta.get("batch_key") or "")
    count_layer = str(meta.get("count_layer") or "counts")

    if not batch_key or batch_key not in method_adata.obs.columns:
        raise KeyError("Dataset batch metadata is missing.")
    if count_layer not in method_adata.layers:
        raise KeyError(f"Raw count layer {count_layer!r} is required for Seurat.")

    mask = _gex_mask(method_adata)
    genes = method_adata.var_names[mask].astype(str)
    counts = method_adata.layers[count_layer][:, mask]
    counts = counts.tocsr() if sparse.issparse(counts) else sparse.csr_matrix(counts)

    if counts.shape != (method_adata.n_obs, int(mask.sum())):
        raise RuntimeError("Unexpected count-matrix shape after GEX filtering.")
    if counts.nnz and np.min(counts.data) < 0:
        raise ValueError("Seurat counts must be non-negative.")
    sample = counts.data[: min(counts.nnz, 100_000)]
    if sample.size and not np.allclose(sample, np.rint(sample), rtol=0.0, atol=1e-6):
        raise ValueError("Seurat requires raw/count-like input; non-integer values were detected.")

    # AnnData is cells x genes; Seurat/Matrix Market input is genes x cells.
    io.mmwrite(workdir / "counts.mtx", counts.T.tocoo())
    pd.DataFrame({"gene": genes}).to_csv(workdir / "genes.csv", index=False)
    pd.DataFrame({"barcode": method_adata.obs_names.astype(str)}).to_csv(
        workdir / "barcodes.csv", index=False
    )
    pd.DataFrame({
        "barcode": method_adata.obs_names.astype(str),
        "BATCH": method_adata.obs[batch_key].astype(str).to_numpy(),
    }).to_csv(workdir / "metadata.csv", index=False)

    print(f"GEX features: {int(mask.sum()):,} / {method_adata.n_vars:,}")
    print(f"Cells: {method_adata.n_obs:,}; batches: {method_adata.obs[batch_key].nunique()}")


def _read_latent(path, expected_barcodes):
    frame = pd.read_csv(path)
    if "barcode" not in frame.columns:
        raise RuntimeError(f"Missing barcode column in {path}")
    barcodes = frame.pop("barcode").astype(str).to_numpy()
    expected = np.asarray(expected_barcodes, dtype=str)
    if not np.array_equal(barcodes, expected):
        raise RuntimeError("Seurat changed cell order; refusing to benchmark a misaligned latent.")
    latent = frame.to_numpy(dtype=np.float32)
    if latent.ndim != 2 or latent.shape[0] != expected.size or latent.shape[1] < 2:
        raise RuntimeError(f"Invalid latent shape: {latent.shape}")
    if not np.isfinite(latent).all():
        raise ValueError("Seurat latent contains NaN or infinite values.")
    return latent, barcodes


def _run_seurat_pair(method_adata, seed, config):
    seed = int(seed)
    cache_key = (seed, int(config["nfeatures"]), int(config["npcs"]), int(config["integration_dims"]))
    if cache_key in SEURAT_CACHE:
        return SEURAT_CACHE[cache_key]

    workdir = SEURAT_ROOT / f"seed_{seed}"
    if workdir.exists():
        shutil.rmtree(workdir)
    workdir.mkdir(parents=True, exist_ok=True)

    _write_seurat_inputs(method_adata, workdir)
    script_path = workdir / "run_seurat.R"
    script_path.write_text(R_SCRIPT, encoding="utf-8")

    command = [
        "Rscript", str(script_path), str(workdir), str(seed),
        str(int(config["nfeatures"])), str(int(config["npcs"])),
        str(int(config["integration_dims"])),
    ]
    print("Running Seurat RPCA + CCA ...")
    subprocess.check_call(command)

    expected = method_adata.obs_names.astype(str).to_numpy()
    rpca = _read_latent(workdir / "seurat_rpca_latent.csv", expected)
    cca = _read_latent(workdir / "seurat_cca_latent.csv", expected)

    provenance = {
        "seurat_runner": script_path,
        "seurat_session_info": workdir / "seurat_session_info.txt",
    }
    SEURAT_CACHE[cache_key] = {
        "rpca": (*rpca, provenance),
        "cca": (*cca, provenance),
    }
    return SEURAT_CACHE[cache_key]


def run_seurat_rpca(method_adata, seed, config):
    latent, barcodes, provenance = _run_seurat_pair(method_adata, seed, config)["rpca"]
    return MethodOutput(
        latent=latent,
        barcodes=barcodes,
        representation_key="X_seurat_rpca",
        provenance_files=provenance,
    )


def run_seurat_cca(method_adata, seed, config):
    latent, barcodes, provenance = _run_seurat_pair(method_adata, seed, config)["cca"]
    return MethodOutput(
        latent=latent,
        barcodes=barcodes,
        representation_key="X_seurat_cca",
        provenance_files=provenance,
    )


In [ ]:
from scrarebench import MethodSpec, benchmark_method

methods = [
    MethodSpec(name="Seurat-RPCA", runner=run_seurat_rpca, config=METHOD_CONFIG),
    MethodSpec(name="Seurat-CCA", runner=run_seurat_cca, config=METHOD_CONFIG),
]

results = {}
for method in methods:
    print(f"\n===== Benchmarking {method.name} =====")
    results[method.name] = benchmark_method(
        adata,
        method,
        seeds=METHOD_SEEDS,
        benchmark_config={"random_state": BENCHMARK_SEED},
        install_dependencies=False,
        finalize=True,
        # Dataset 0 is large. The method runner never mutates adata, so reuse it
        # instead of creating multiple full AnnData copies in Colab.
        copy_input_per_seed=False,
        overwrite=True,
    )


In [ ]:
from IPython.display import display
import pandas as pd

summary = pd.concat(
    [result.summary().assign(method=name) for name, result in results.items()],
    ignore_index=True,
)
display(summary[["method", "method_seed", "section", "metric", "value"]].round(5))

for name, result in results.items():
    print(f"\n{name}")
    print("Seeds:", result.seeds)
    print("Report:", result.report_path)
    print("Archive:", result.archive_path)
    print("Summary JSON:", result.summary_path)


In [ ]:
# Download all Seurat benchmark output archives as one ZIP
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile
from google.colab import files

download_items = {}

for method_name, result in results.items():
    archive_path = Path(result.archive_path)

    if not archive_path.exists():
        raise FileNotFoundError(
            f"Output archive for {method_name} was not found: {archive_path}\n"
            "Make sure benchmark_method(..., finalize=True) completed successfully."
        )

    download_items[method_name] = archive_path

    print(
        f"{method_name}: {archive_path.name} "
        f"({archive_path.stat().st_size / (1024**2):.2f} MB)"
    )

combined_archive = Path("/content/scRareBench_Seurat_Dataset0_outputs.zip")

with ZipFile(combined_archive, "w", compression=ZIP_DEFLATED) as zf:
    for method_name, archive_path in download_items.items():
        zf.write(
            archive_path,
            arcname=f"{method_name}/{archive_path.name}",
        )

print(f"\nDownloading: {combined_archive.name}")
print(f"Total size: {combined_archive.stat().st_size / (1024**2):.2f} MB")

files.download(str(combined_archive))